# Visualization

This notebook serves to visualize the results of the models.

In [1]:
import os
import shutil
import h5py
import numpy as np
import sys
import pandas as pd
import json
import matplotlib.pyplot as plt
%matplotlib inline
import importlib

sys.path.append("..")
sys.path.append("../code")
sys.path.append(os.path.join("..", 'models','Pointnet_Pointnet2_pytorch', 'models'))

from dataset import PCExtrusionSegmentationDataset
from models.DeepCAD.cadlib.visualize import vec2CADsolid
from OCC.Core.BRepCheck import BRepCheck_Analyzer
from OCC.Extend.DataExchange import write_step_file
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.StlAPI import StlAPI_Writer
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh
from models.DeepCAD.cadlib.extrude import CADSequence
from models.DeepCAD.cadlib.visualize import create_CAD
from models.DeepCAD.cadlib.visualize import CADsolid2pc
from models.DeepCAD.utils.pc_utils import write_ply
import open3d as o3d
from metrics import ClassificationRunningScore
import torch

## Extrusion Segmentation

### Visualization

In [53]:
def get_trained_segmentation_pn2(model_path):
   
    model_name = 'pointnet2_sem_seg_msg'
    model = importlib.import_module(model_name)
    num_classes = 10
    classifier = model.get_model(num_classes)
    
    trained_model = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)
    state_dict = trained_model['model_state_dict']
    classifier.load_state_dict(state_dict)
    config = trained_model['config']

    return classifier, config
    

In [54]:
import open3d as o3d
import matplotlib.pyplot as plt

def visualize_labeled_pc(points, labels):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)

    colors = plt.cm.tab10(labels / labels.max())[:, :3] 
    pcd.colors = o3d.utility.Vector3dVector(colors)

    o3d.visualization.draw_geometries([pcd])

In [55]:
def infer_segmentation_pn2(model, pc, show=True):
    pc = pc.unsqueeze(0)
    pc = pc.transpose(2, 1)
    pred_logits, _ = model(pc)
    pred_logits = pred_logits.data.view(-1, 10)
    pred = pred_logits.max(1)[1]
    return pred

In [56]:
run_name = "partseg_overfit_on_first_5"
model_path = os.path.join("..", "models", "trained_models", run_name, "ckpt_20.pth")
classifier, config = get_trained_segmentation_pn2(model_path)
config

{'learning_rate': 0.001,
 'batch_size': 5,
 'max_epochs': 20,
 'optimizer': 'Adam',
 'model_type': 'pointnet2_sem_seg_msg',
 'save_interval': 20,
 'early_stopping': 20,
 'start_time': '2025-07-15_16-55-55',
 'lr_type': 'step',
 'gpu': False,
 'final_epoch': 20,
 'training_time_min': 11.3}

In [57]:
train_dataset = PCExtrusionSegmentationDataset("../data", 'train', use_normals=False, verbose=False)
#val_dataset = PCExtrusionSegmentationDataset("../data", 'validation', use_normals=False, verbose=False)
#test_dataset = PCExtrusionSegmentationDataset("../data", 'test', use_normals=False, verbose=False)

In [58]:
index = 0
data = train_dataset[index]
pc = data['pc']
label = data['label']

In [8]:
visualize_labeled_pc(pc, label)
pred_labels = infer_segmentation_pn2(classifier, pc)
visualize_labeled_pc(pc, pred_labels)

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [9]:
for i in range(0):
    data = train_dataset[i]
    pc = data['pc']
    label = data['label']
    visualize_labeled_pc(pc, label)
    pred_labels = infer_segmentation_pn2(classifier, pc)
    visualize_labeled_pc(pc, pred_labels)

In [59]:
def save_pc_with_labels_for_blender(points, labels, name):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points.numpy())
    
    colors = np.zeros((points.shape[0], 3))
    color_map = np.array([
        [0.894, 0.102, 0.110],  # class 0
        [0.216, 0.494, 0.722],  # class 1
        [0.302, 0.686, 0.290],  # class 2
        [0.596, 0.306, 0.639],  # class 3
        [1.000, 0.498, 0.0],    # class 4
        [1.000, 1.000, 0.2],    # class 5
        [0.651, 0.337, 0.157],  # class 6
        [0.969, 0.506, 0.749],  # class 7
        [0.6,   0.6,   0.6],    # class 8
        [0.1,   0.1,   0.1],    # class 9
    ])
    colors = color_map[labels.numpy()]
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    o3d.io.write_point_cloud(os.path.join("examples", name + ".ply"), pcd)

In [11]:
save_pc_with_labels_for_blender(pc, pred_labels, "pred_pc_2")

## End-to-end pipeline: Inference

### Imports

In [2]:
#e2e pipeline
from pn2_deepcad import Config
from models.DeepCAD.trainer.loss import CADLoss
from dataset import PointCloudEmbeddingSequenceDataset
from models.DeepCAD.cadlib.macro import CMD_ARGS_MASK, ALL_COMMANDS, EOS_IDX, SOL_IDX, EXT_IDX, ARC_IDX, N_ARGS
from models.DeepCAD.utils import read_ply
import random
from scipy.spatial import cKDTree as KDTree
from OCC.Core.Bnd import Bnd_Box
from OCC.Core.BRepBndLib import brepbndlib_Add
from OCC.Extend.DataExchange import write_stl_file

from models.DeepCAD.config.configAE import ConfigAE
from models.DeepCAD.trainer.trainerAE import TrainerAE

### Functions for e2e pipeline

In [3]:
def save_pc(pc, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pc)
    o3d.io.write_point_cloud(path, pcd)

In [4]:
def inplace_relu(m):
    classname = m.__class__.__name__
    if classname.find('ReLU') != -1:
        m.inplace=True

def logits2vec(outputs, refill_pad=True, to_numpy=True):
    """network outputs (logits) to final CAD vector"""
    out_command = torch.argmax(torch.softmax(outputs['command_logits'], dim=-1), dim=-1)  # (N, S)
    out_args = torch.argmax(torch.softmax(outputs['args_logits'], dim=-1), dim=-1) - 1  # (N, S, N_ARGS)
    if refill_pad: # fill all unused element to -1
        mask = ~torch.tensor(CMD_ARGS_MASK).bool()[out_command.long()]
        out_args[mask] = -1

    out_cad_vec = torch.cat([out_command.unsqueeze(-1), out_args], dim=-1)
    if to_numpy:
        out_cad_vec = out_cad_vec.detach().cpu().numpy()
    return out_cad_vec

def calculate_ACC(results):

    TOLERANCE = 3
    
    # accuracy w.r.t. each command type
    each_cmd_cnt = np.zeros((len(ALL_COMMANDS),))
    each_cmd_acc = np.zeros((len(ALL_COMMANDS),))

    # accuracy w.r.t each parameter
    args_mask = CMD_ARGS_MASK.astype(np.float32)
    N_ARGS = args_mask.shape[1]
    each_param_cnt = np.zeros([*args_mask.shape])
    each_param_acc = np.zeros([*args_mask.shape])

    B = results["tgt_commands"].shape[0]

    for i in range(B): # for each sample in the batch
        seq_length = list(results["tgt_commands"][i]).index(EOS_IDX)
        out_cmd = results["pred"][i,:seq_length,0]
        gt_cmd = results["tgt_commands"][i, :seq_length].numpy()
    
        out_param = results["pred"][i,:seq_length,1:]
        gt_param = results["tgt_args"][i, :seq_length].numpy()

        cmd_acc = (out_cmd == gt_cmd).astype(np.int32)
        param_acc = []
              
        for j in range(len(gt_cmd)):
            cmd = gt_cmd[j]
            each_cmd_cnt[cmd] += 1
            each_cmd_acc[cmd] += cmd_acc[j]
            if cmd in [SOL_IDX, EOS_IDX]:
                continue
        
            if out_cmd[j] == gt_cmd[j]: # NOTE: only account param acc for correct cmd
                tole_acc = (np.abs(out_param[j] - gt_param[j]) < TOLERANCE).astype(np.int32)
                
                # filter param that do not need tolerance (i.e. requires strictly equal)
                if cmd == EXT_IDX:
                    tole_acc[-2:] = (out_param[j] == gt_param[j]).astype(np.int32)[-2:]
                elif cmd == ARC_IDX:
                    tole_acc[3] = (out_param[j] == gt_param[j]).astype(np.int32)[3]

                valid_param_acc = tole_acc[args_mask[cmd].astype(bool)].tolist()
                param_acc.extend(valid_param_acc)
                each_param_cnt[cmd, np.arange(N_ARGS)] += 1
                each_param_acc[cmd, np.arange(N_ARGS)] += tole_acc

    # acc of each parameter type
    each_param_acc = each_param_acc * args_mask
    each_param_cnt = each_param_cnt * args_mask

    return {"each_cmd_acc": each_cmd_acc, # per cmd number of correct 
            "each_cmd_cnt": each_cmd_cnt, # per cmd count
            "each_param_acc": each_param_acc, # per param number of correct
            "each_param_cnt": each_param_cnt} # per param count

def normalize_pc(points):
    scale = np.max(np.abs(points))
    points = points / scale
    return points

def chamfer_dist(gt_points, gen_points, offset=0, scale=1):
    gen_points = gen_points / scale - offset

    # one direction
    gen_points_kd_tree = KDTree(gen_points)
    one_distances, one_vertex_ids = gen_points_kd_tree.query(gt_points)
    gt_to_gen_chamfer = np.mean(np.square(one_distances))

    # other direction
    gt_points_kd_tree = KDTree(gt_points)
    two_distances, two_vertex_ids = gt_points_kd_tree.query(gen_points)
    gen_to_gt_chamfer = np.mean(np.square(two_distances))

    return gt_to_gen_chamfer + gen_to_gt_chamfer

def calculate_CD(vec_pred, gt_pc_path, vec_target, data_id):
    seq_len = vec_target[:,0].tolist().index(EOS_IDX)
    vec_pred = vec_pred.squeeze()[:seq_len]

    try:
        shape = vec2CADsolid(vec_pred) # out vec only contains until target seq length
    except Exception as e:
        return float('nan') # Create CAD failed
    
    try:
        out_pc = CADsolid2pc(shape, 2000, data_id) # 2000 is the number of sampled points
    except Exception as e:
        return float('nan') # Create PC failed

    if np.max(np.abs(out_pc)) > 2: # normalize out-of-bound data
        out_pc = normalize_pc(out_pc)

    gt_pc = read_ply(gt_pc_path)
    sample_idx = random.sample(list(range(gt_pc.shape[0])), 2000)
    gt_pc = gt_pc[sample_idx]

    cd = chamfer_dist(gt_pc, out_pc)
    return cd

In [5]:
def get_trained_pn2(model_path):
    saved_model = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)
    config = saved_model['config']
    
    for key, value in config.items():
        print(f"{key}: {value}")
        
    cfg = Config()
    model_name = "pn2_deepcad"
    model = importlib.import_module(model_name)
    classifier = model.get_pn2_deepcad_model(cfg, normal_channel=False)
    criterion = CADLoss(cfg)
    classifier.apply(inplace_relu)
    state_dict = saved_model['model_state_dict']
    classifier.load_state_dict(state_dict)
    classifier.eval()
    print(f'\nLoaded state dict from {model_path}.')
    return classifier, criterion
    

In [6]:
def infer_pn2(data, classifier, criterion):
    with torch.no_grad():
        id = data['id']
        pc = data['pc']
        pc_path = f"examples/{id}.ply"
        save_pc(pc, pc_path)
        print(f"Saved pc to {pc_path}")
        pc = pc.unsqueeze(0)
        sequence = data['tgt_vec']
        sequence = sequence.unsqueeze(0)
        
        
    
        pc = pc.transpose(2, 1)
        output = classifier(pc)
    
        tgt_commands = sequence[:, :, 0]
        tgt_args = sequence[:, :, 1:]
    
        seq_length = list(tgt_commands[0]).index(EOS_IDX)
    
        output["tgt_commands"] = tgt_commands
        output["tgt_args"] = tgt_args
    
        loss_dict = criterion(output)
    
        cmd_loss = loss_dict['loss_cmd'].detach().cpu().item()
        args_loss = loss_dict['loss_args'].detach().cpu().item()
    
        batch_out_vec = logits2vec(output)
        pred_commands = batch_out_vec[:, :, 0]
        pred_args = batch_out_vec[:, :, 1:]
        pred_seq_length = list(pred_commands[0]).index(EOS_IDX)
    
        metrics = calculate_ACC({"tgt_commands": tgt_commands,
                                "tgt_args": tgt_args,
                                "pred": batch_out_vec})
        gt_pc_path = os.path.join("../data", "pc_from_vec", id[:4], id + ".ply")
        if not os.path.exists(gt_pc_path):
            cd = float('nan')
        else:
            cd = calculate_CD(batch_out_vec, gt_pc_path, sequence[0].detach().cpu().numpy(), id)

        ## STL creation
        shape = vec2CADsolid(batch_out_vec.squeeze()[:seq_length])
        bbox = Bnd_Box()
        brepbndlib_Add(shape, bbox)
        if bbox.IsVoid():
            raise ValueError("box check failed")
        name = id
        stl_path = "examples/e2e_{}.stl".format(name)
        write_stl_file(shape, stl_path, linear_deflection=0.05, angular_deflection=0.05)
        print(f"Saved stl file to {stl_path}")
        ###
    
        sample_cmd_acc = np.sum(metrics["each_cmd_acc"]) / np.sum(metrics["each_cmd_cnt"] + 1e-6)
        sample_param_acc = np.sum(metrics["each_param_acc"]) / np.sum(metrics["each_param_cnt"] + 1e-6)
    
        COMMAND_NAMES = ['L', 'A', 'C', 'EOS', 'S', 'E']
        tgt_args_list = []
        pred_args_list = []
        for k, cmd in enumerate(tgt_commands.squeeze()[:seq_length].tolist()):
            if not k == seq_length:
                tgt_args_list.append(f"{COMMAND_NAMES[cmd]}")
            params = tgt_args.squeeze()[k]
            selected_args = params[torch.tensor(CMD_ARGS_MASK[cmd]).bool()]
            tgt_args_list.extend(selected_args.tolist())
        for k, cmd in enumerate(pred_commands.squeeze()[:seq_length].tolist()):
            if not k == pred_seq_length:
                pred_args_list.append(f"{COMMAND_NAMES[cmd]}")
            params = pred_args.squeeze()[k]
            selected_args = params[torch.tensor(CMD_ARGS_MASK[cmd]).bool()]
            pred_args_list.extend(selected_args.tolist())
    
        tgt_commands = tgt_commands.squeeze()[:seq_length].tolist()
        pred_commands = pred_commands.squeeze()[:pred_seq_length].tolist()

    results = {
        "id": id,                   # str
        "cmd_acc": sample_cmd_acc,         # float
        "param_acc": sample_param_acc,     # float
        "cd": cd,                   # float
        "tgt_commands": tgt_commands, # torch.Tensor (60,)
        "pred_commands": pred_commands,
        "tgt_args": tgt_args_list,         # torch.Tensor (60, 16)
        "pred_args": pred_args_list,
        "seq_len": seq_length,        # int
        "pred_seq_len": pred_seq_length, # int
        "cmd_count": metrics["each_cmd_cnt"].astype(int),     # torch.Tensor (6,)
        "cmd_correct": metrics["each_cmd_acc"].astype(int), # torch.Tensor (6,)
        "cmd_count_total": np.sum(metrics["each_cmd_cnt"]).astype(int), # int
        "cmd_correct_total": np.sum(metrics["each_cmd_acc"]).astype(int), # int
        "per_cmd_param_count": metrics["each_param_cnt"].astype(int), # torch.Tensor (6×16)
        "per_cmd_param_correct": metrics["each_param_acc"].astype(int), # torch.Tensor (6×16)
        "param_count_total": np.sum(metrics["each_param_cnt"]).astype(int), # int
        "param_correct_total": np.sum(metrics["each_param_acc"]).astype(int), # int
        "cmd_loss": cmd_loss,              # float
        "param_loss": args_loss,          # float
        "total_loss": cmd_loss + args_loss, # float
    }
    return results
    

### Functions for 2-stage pipeline

In [7]:
def load_pointnet(model_path):
    
    saved_model = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)
    state_dict = saved_model['model_state_dict']
    config = saved_model['config']
    if 'module.' in next(iter(state_dict)):
        monitor.log_and_print("Model was saved wrapped in nn.DataParallel.\nRemoving 'module.' from state dict.")
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

    sys.path.append(os.path.join('..', 'models','Pointnet_Pointnet2_pytorch', 'models'))

    normal_channel = False
    model = importlib.import_module(config['model_type'])
    if 'architecture' in config:
        if config['architecture'] == 'own':
            classifier = model.get_model(256, normal_channel=normal_channel)
        elif config['architecture'] == "copy_author":
            classifier = model.get_model_copy_author(256, normal_channel=normal_channel)
        elif config['architecture'] == "tanh":
            classifier = model.get_model_tanh(256, normal_channel=normal_channel)
    else:
        classifier = model.get_model(256, normal_channel=False)
    criterion = model.get_loss_mse()
    classifier.apply(inplace_relu) 
    
    
    classifier.load_state_dict(state_dict)
    classifier.eval()
    print(f"Loading PointNet++ from {os.path.abspath(model_path)}")
    return classifier, criterion
    

In [8]:
def load_deepcad(cfg):
    tr_agent = TrainerAE(cfg)
    tr_agent.load_ckpt(cfg.ckpt)
    tr_agent.net.eval()
    return tr_agent

In [9]:
def infer_pointnet(data, pn_model, pn_criterion):
    pc = data['pc']
    pc = pc.unsqueeze(0)
    z = data['z']
    seq_target = data['tgt_vec']
    
    with torch.no_grad():
        pc = pc.transpose(2, 1)
        z_pred, _ = pn_model(pc)
        z_pred = z_pred.squeeze()
        loss = pn_criterion(z_pred, z)
        print(f"Avg. MSE-Loss: {loss.detach().item():.8f}")
        return z_pred, seq_target

In [10]:
def infer_deepcad(pred, tr_agent, data):
    
    with torch.no_grad():

        sequence = data['tgt_vec']
        id = data['id']
        
        sequence = sequence.unsqueeze(0)
        tgt_commands = sequence[:, :, 0]
        tgt_args = sequence[:, :, 1:]
        pred = pred.unsqueeze(0).unsqueeze(0)
        output = tr_agent.decode(pred)

        seq_length = list(tgt_commands[0]).index(EOS_IDX)

        output["tgt_commands"] = sequence[:, :, 0] 
        output["tgt_args"] = sequence[:, :, 1:]

        loss_dict = tr_agent.loss_func(output)
        cmd_loss = loss_dict['loss_cmd']
        args_loss = loss_dict['loss_args']
        
        batch_out_vec = tr_agent.logits2vec(output)

        pred_commands = batch_out_vec[:, :, 0]
        pred_args = batch_out_vec[:, :, 1:]
        pred_seq_length = list(pred_commands[0]).index(EOS_IDX)
    
        metrics = calculate_ACC({"tgt_commands": tgt_commands,
                                "tgt_args": tgt_args,
                                "pred": batch_out_vec})
        gt_pc_path = os.path.join("../data", "pc_from_vec", id[:4], id + ".ply")
        if not os.path.exists(gt_pc_path):
            cd = float('nan')
        else:
            cd = calculate_CD(batch_out_vec, gt_pc_path, sequence[0].detach().cpu().numpy(), id)

        ## STL creation
        shape = vec2CADsolid(batch_out_vec.squeeze()[:seq_length])
        bbox = Bnd_Box()
        brepbndlib_Add(shape, bbox)
        if bbox.IsVoid():
            raise ValueError("box check failed")
        name = id
        stl_path = "examples/two_stage_{}.stl".format(name)
        write_stl_file(shape, stl_path, linear_deflection=0.05, angular_deflection=0.05)
        print(f"Saved stl file to {stl_path}")
        ###
    
        sample_cmd_acc = np.sum(metrics["each_cmd_acc"]) / np.sum(metrics["each_cmd_cnt"] + 1e-6)
        sample_param_acc = np.sum(metrics["each_param_acc"]) / np.sum(metrics["each_param_cnt"] + 1e-6)
    
        COMMAND_NAMES = ['L', 'A', 'C', 'EOS', 'S', 'E']
        tgt_args_list = []
        pred_args_list = []
        for k, cmd in enumerate(tgt_commands.squeeze()[:seq_length].tolist()):
            if not k == seq_length:
                tgt_args_list.append(f"{COMMAND_NAMES[cmd]}")
            params = tgt_args.squeeze()[k]
            selected_args = params[torch.tensor(CMD_ARGS_MASK[cmd]).bool()]
            tgt_args_list.extend(selected_args.tolist())
        for k, cmd in enumerate(pred_commands.squeeze()[:seq_length].tolist()):
            if not k == pred_seq_length:
                pred_args_list.append(f"{COMMAND_NAMES[cmd]}")
            params = pred_args.squeeze()[k]
            selected_args = params[torch.tensor(CMD_ARGS_MASK[cmd]).bool()]
            pred_args_list.extend(selected_args.tolist())
    
        tgt_commands = tgt_commands.squeeze()[:seq_length].tolist()
        pred_commands = pred_commands.squeeze()[:pred_seq_length].tolist()

    results = {
        "id": id,                   # str
        "cmd_acc": sample_cmd_acc,         # float
        "param_acc": sample_param_acc,     # float
        "cd": cd,                   # float
        "tgt_commands": tgt_commands, # torch.Tensor (60,)
        "pred_commands": pred_commands,
        "tgt_args": tgt_args_list,         # torch.Tensor (60, 16)
        "pred_args": pred_args_list,
        "seq_len": seq_length,        # int
        "pred_seq_len": pred_seq_length, # int
        "cmd_count": metrics["each_cmd_cnt"].astype(int),     # torch.Tensor (6,)
        "cmd_correct": metrics["each_cmd_acc"].astype(int), # torch.Tensor (6,)
        "cmd_count_total": np.sum(metrics["each_cmd_cnt"]).astype(int), # int
        "cmd_correct_total": np.sum(metrics["each_cmd_acc"]).astype(int), # int
        "per_cmd_param_count": metrics["each_param_cnt"].astype(int), # torch.Tensor (6×16)
        "per_cmd_param_correct": metrics["each_param_acc"].astype(int), # torch.Tensor (6×16)
        "param_count_total": np.sum(metrics["each_param_cnt"]).astype(int), # int
        "param_correct_total": np.sum(metrics["each_param_acc"]).astype(int), # int
        "cmd_loss": cmd_loss,              # float
        "param_loss": args_loss,          # float
        "total_loss": cmd_loss + args_loss, # float
    }
    return results

### Start

In [92]:
run_name = "complex_run"
e2e_model_path = os.path.join("..", "models", "trained_models", run_name, "best.pth")
e2e_classifier, e2e_criterion = get_trained_pn2(e2e_model_path)
test_dataset = PointCloudEmbeddingSequenceDataset("../data", 'test', use_normals=False)

learning_rate: 0.0001
batch_size: 192
max_epochs: 1000
optimizer: Adam
model_type: pn2_deepcad
save_interval: 50
early_stopping: 50
start_time: 2025-07-09_18-06-28
lr_type: step_adv
gpu: True
final_epoch: 185
training_time_min: 2627.04

Loaded state dict from ../models/trained_models/complex_run/best.pth.


In [93]:
data = test_dataset[7640]
results_e2e = infer_pn2(data, e2e_classifier, e2e_criterion)

Saved pc to examples/00270325.ply
Saved stl file to examples/e2e_00270325.stl


In [94]:
for k,v in results_e2e.items():
    print(f"{k:<19}: {v}")

id                 : 00270325
cmd_acc            : 0.8749996718751231
param_acc          : 0.4999980000080004
cd                 : 0.009874751200904433
tgt_commands       : [4, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 5]
pred_commands      : [4, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 5]
tgt_args           : ['S', 'L', 152, 87, 'L', 199, 87, 'L', 223, 128, 'L', 199, 169, 'L', 152, 169, 'L', 128, 128, 'S', 'L', 161, 102, 'L', 176, 102, 'L', 190, 102, 'L', 205, 128, 'L', 190, 154, 'L', 161, 154, 'L', 146, 128, 'E', 128, 128, 128, 83, 128, 128, 89, 224, 128, 0, 0]
pred_args          : ['S', 'L', 152, 87, 'L', 199, 87, 'L', 223, 128, 'L', 199, 169, 'L', 152, 169, 'L', 128, 128, 'S', 'L', 152, 128, 'L', 199, 87, 'L', 223, 128, 'L', 199, 164, 'L', 152, 169, 'L', 152, 128, 'E', 128, 128, 128, 91, 128, 128, 96, 224, 128, 0, 0]
seq_len            : 16
pred_seq_len       : 15
cmd_count          : [13  0  0  0  2  1]
cmd_correct        : [12  0  0  0  2  0]
cmd_count_total    : 16
cmd_correct_tot

In [95]:
two_stage_run_name = "slow_run_2"
two_stage_model_path = os.path.join("..", "models", "trained_models", two_stage_run_name, "best.pth")
pn_classifier, pn_criterion = load_pointnet(two_stage_model_path)
cfg_ae = ConfigAE('test', model_path="../data/latent", parse=False)
deepcad = load_deepcad(cfg_ae)

Loading PointNet++ from /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/models/trained_models/slow_run_2/best.pth
Loading checkpoint from /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/data/latent/pretrained/model/ckpt_epoch1000.pth ...


In [96]:
z_pred, tgt_vec = infer_pointnet(data, pn_classifier, pn_criterion)

Avg. MSE-Loss: 0.07083814


In [97]:
lol = infer_deepcad(z_pred, deepcad, data)

Saved stl file to examples/two_stage_00270325.stl


In [98]:
for k,v in lol.items():
    print(f"{k:<19}: {v}")

id                 : 00270325
cmd_acc            : 0.49999981250007036
param_acc          : 0.9999920000639992
cd                 : 0.02581761018028983
tgt_commands       : [4, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 5]
pred_commands      : [4, 0, 0, 0, 0, 0, 0, 4, 2, 5]
tgt_args           : ['S', 'L', 152, 87, 'L', 199, 87, 'L', 223, 128, 'L', 199, 169, 'L', 152, 169, 'L', 128, 128, 'S', 'L', 161, 102, 'L', 176, 102, 'L', 190, 102, 'L', 205, 128, 'L', 190, 154, 'L', 161, 154, 'L', 146, 128, 'E', 128, 128, 128, 83, 128, 128, 89, 224, 128, 0, 0]
pred_args          : ['S', 'L', 152, 87, 'L', 199, 87, 'L', 223, 128, 'L', 199, 169, 'L', 152, 169, 'L', 128, 128, 'S', 'C', 176, 128, 22, 'E', 128, 128, 128, 108, 128, 128, 96, 224, 128, 0, 0, 'EOS', 'EOS', 'EOS', 'EOS', 'EOS']
seq_len            : 16
pred_seq_len       : 10
cmd_count          : [13  0  0  0  2  1]
cmd_correct        : [6 0 0 0 2 0]
cmd_count_total    : 16
cmd_correct_total  : 8
per_cmd_param_count: [[6 6 0 0 0 0 0 0 0 0 0 0 

## End-to-end pipeline: Test Metrics

In [87]:
def show_sample(df, idx):
    with pd.option_context('display.max_colwidth', None):
        print(df.iloc[idx])

In [88]:
def show_metrics(df):

    ALL_COMMANDS = ['Line', 'Arc', 'Circle', 'EOS', 'SOL', 'Ext']
    cmd_count = df["cmd_count"]
    cmd_correct = df["cmd_correct"].to_numpy()
    cmd_count_total = df["cmd_count_total"].tolist()
    cmd_correct_total = df["cmd_correct_total"].tolist()
    
    cmd_acc = np.sum(cmd_correct_total)/np.sum(cmd_count_total)
    each_cmd_acc = np.sum(cmd_correct)/(np.sum(cmd_count) + 1e-8)
    
    ALL_ARGS = ["x", "y", "alpha", "f", "r", "theta", "phi", "gamma", "p_x", "p_y", "p_z", "s", "e_1", "e_2", "b", "u"]
    per_cmd_param_count = df["per_cmd_param_count"]
    per_cmd_param_correct = df["per_cmd_param_correct"]
    param_count_total = df["param_count_total"].tolist()
    param_correct_total = df["param_correct_total"].tolist()
    
    param_acc = np.sum(param_correct_total)/np.sum(param_count_total)
    each_param_acc = np.sum(per_cmd_param_correct)/(np.sum(per_cmd_param_count) + 1e-8)
    
    
    
    print(f"Command Accuracy  : {cmd_acc * 100:.2f}%")
    print(f"Parameter Accuracy: {param_acc * 100:.2f}%")
    print()
    for i, cmd in enumerate(ALL_COMMANDS):
        print(f"{cmd:<6}: {each_cmd_acc[i] * 100:.2f}%")
    
    print()
    for i, cmd in enumerate(each_cmd_acc):
        print(ALL_COMMANDS[i])
        for j, param in enumerate(each_param_acc[i]):
            if param != 0:
                print(f"{ALL_ARGS[j]:<5}: {param * 100:.2f}%")
        print()
    
    cd = df["cd"].tolist()
    median_cd = np.nanmedian(cd)
    print(f"Median CD * 1e3: {median_cd * 1e3:.2f}")
    ir = np.isnan(cd).sum() / len(cd)
    print(f"Invalid Ratio  : {ir * 100:.2f}%")

In [55]:
run_name = "complex_run"
df = pd.read_pickle(os.path.join("..", "models", "trained_models", run_name, "test_sample_results.pkl"))

In [56]:
df

,id,set_id,cmd_acc,param_acc,cd,tgt_commands,pred_commands,tgt_args,pred_args,seq_len,...,cmd_correct,cmd_count_total,cmd_correct_total,per_cmd_param_count,per_cmd_param_correct,param_count_total,param_correct_total,cmd_loss,param_loss,total_loss
0,00250456,0,0.999999,0.999995,0.004115,"[4, 0, 0, 0, 0, 5]","[4, 0, 0, 0, 0, 5]","[S, L, 223, 128, L, 223, 223, L, 128, 223, L, ...","[S, L, 223, 128, L, 223, 223, L, 128, 223, L, ...",6,...,"[4, 0, 0, 0, 1, 1]",6,6,"[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",19,19,0.035257,0.198660,0.233917
1,00440420,1,0.999999,0.555554,0.016934,"[4, 0, 0, 0, 0, 0, 0, 0, 0, 5]","[4, 0, 0, 0, 0, 0, 0, 0, 0, 5]","[S, L, 150, 106, L, 201, 106, L, 223, 128, L, ...","[S, L, 136, 99, L, 194, 99, L, 223, 128, L, 22...",10,...,"[8, 0, 0, 0, 1, 1]",10,10,"[[8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[4, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",27,15,0.086120,4.028588,4.114707
2,00819758,2,0.785714,0.529409,NaN,"[4, 0, 0, 0, 0, 4, 2, 4, 2, 4, 2, 4, 2, 5]","[4, 0, 0, 0, 0, 0, 0, 4, 2, 4, 2, 4, 2, 4]","[S, L, 136, 108, L, 215, 108, L, 223, 128, L, ...","[S, L, 134, 118, L, 223, 118, L, 223, 128, L, ...",14,...,"[4, 0, 3, 0, 4, 0]",14,11,"[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",17,9,0.478164,3.858808,4.336972
3,00239323,3,0.899999,0.555553,NaN,"[4, 1, 0, 1, 0, 4, 2, 4, 2, 5]","[4, 1, 0, 1, 0, 4, 2, 4, 2, 4, 2, 5, 2, 5]","[S, A, 176, 128, 128, 1, L, 176, 199, A, 128, ...","[S, A, 179, 128, 128, 1, L, 185, 191, A, 128, ...",10,...,"[2, 2, 2, 0, 3, 0]",10,9,"[[2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",18,10,0.843998,2.469708,3.313707
4,00420729,4,0.999999,0.789470,0.003497,"[4, 0, 0, 0, 0, 5]","[4, 0, 0, 0, 0, 5]","[S, L, 223, 128, L, 223, 179, L, 128, 179, L, ...","[S, L, 223, 128, L, 223, 176, L, 128, 176, L, ...",6,...,"[4, 0, 0, 0, 1, 1]",6,6,"[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[4, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",19,15,0.055654,1.689752,1.745406
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8033,00068472,8033,0.999999,0.999995,0.002185,"[4, 0, 0, 0, 0, 5]","[4, 0, 0, 0, 0, 5]","[S, L, 223, 128, L, 223, 223, L, 128, 223, L, ...","[S, L, 223, 128, L, 223, 223, L, 128, 223, L, ...",6,...,"[4, 0, 0, 0, 1, 1]",6,6,"[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",19,19,0.019508,0.455970,0.475478
8034,00273246,8034,1.000000,0.967739,0.000191,"[4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5]","[4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5]","[S, L, 223, 128, L, 223, 152, L, 204, 171, L, ...","[S, L, 223, 128, L, 223, 152, L, 204, 171, L, ...",12,...,"[10, 0, 0, 0, 1, 1]",12,12,"[[10, 10, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[[10, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0...",31,30,0.001946,0.354055,0.356002
8035,00910275,8035,0.999999,0.999995,0.002164,"[4, 0, 0, 0, 0, 5]","[4, 0, 0, 0, 0, 5]","[S, L, 223, 128, L, 223, 223, L, 128, 223, L, ...","[S, L, 223, 128, L, 223, 223, L, 128, 223, L, ...",6,...,"[4, 0, 0, 0, 1, 1]",6,6,"[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",19,19,0.023031,0.156273,0.179304
8036,00736413,8036,0.333333,0.999968,0.002138,"[4, 2, 5, 4, 2, 5]","[4, 2, 4, 2, 5]","[S, C, 176, 128, 48, E, 128, 128, 128, 32, 128...","[S, C, 176, 128, 48, S, C, 176, 128, 16, E, 12...",6,...,"[0, 0, 1, 0, 1, 0]",6,2,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",3,3,1.352926,1.375347,2.728273


In [57]:
# show_sample(df, 4)

In [58]:
show_metrics(df)

NameError: name 'show_metrics' is not defined

In [64]:
sorted_df = df.sort_values(by="cd") 
sorted_df[3800:3850]

,id,set_id,cmd_acc,param_acc,cd,tgt_commands,pred_commands,tgt_args,pred_args,seq_len,...,cmd_correct,cmd_count_total,cmd_correct_total,per_cmd_param_count,per_cmd_param_correct,param_count_total,param_correct_total,cmd_loss,param_loss,total_loss
3977,00184790,3977,0.999999,0.821426,0.005517,"[4, 2, 5, 4, 2, 5]","[4, 2, 5, 4, 2, 5]","[S, C, 176, 128, 47, E, 128, 128, 128, 93, 128...","[S, C, 176, 128, 48, E, 128, 128, 128, 94, 128...",6,...,"[0, 0, 2, 0, 2, 2]",6,6,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",28,23,0.101310,1.783383,1.884693
2337,00618354,2337,0.999999,0.789470,0.005518,"[4, 0, 0, 0, 0, 5]","[4, 0, 0, 0, 0, 5]","[S, L, 198, 128, L, 198, 223, L, 128, 223, L, ...","[S, L, 191, 128, L, 191, 223, L, 128, 223, L, ...",6,...,"[4, 0, 0, 0, 1, 1]",6,6,"[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[2, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",19,15,0.006312,2.274274,2.280587
903,00367823,903,0.999999,0.947364,0.005522,"[4, 0, 0, 0, 0, 5]","[4, 0, 0, 0, 0, 5]","[S, L, 191, 128, L, 191, 223, L, 128, 223, L, ...","[S, L, 191, 128, L, 191, 223, L, 128, 223, L, ...",6,...,"[4, 0, 0, 0, 1, 1]",6,6,"[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",19,18,0.030942,0.787905,0.818847
7758,00293501,7758,0.999999,0.956518,0.005524,"[4, 0, 0, 0, 0, 0, 0, 5]","[4, 0, 0, 0, 0, 0, 0, 5]","[S, L, 153, 87, L, 200, 88, L, 223, 130, L, 19...","[S, L, 152, 87, L, 199, 87, L, 223, 128, L, 19...",8,...,"[6, 0, 0, 0, 1, 1]",8,8,"[[6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",23,22,0.004646,2.736879,2.741525
1518,00241553,1518,0.999999,0.909087,0.005527,"[4, 0, 0, 0, 0, 4, 2, 5]","[4, 0, 0, 0, 0, 4, 2, 5]","[S, L, 223, 128, L, 223, 160, L, 128, 160, L, ...","[S, L, 223, 128, L, 223, 162, L, 128, 156, L, ...",8,...,"[4, 0, 1, 0, 2, 1]",8,8,"[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[4, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",22,20,0.029811,1.162730,1.192540
4492,00594097,4492,0.777777,0.842101,0.005529,"[4, 0, 0, 0, 0, 5, 4, 2, 5]","[4, 0, 0, 0, 0, 5, 4, 0, 0, 0, 0, 5]","[S, L, 211, 128, L, 211, 223, L, 128, 223, L, ...","[S, L, 223, 128, L, 223, 223, L, 128, 223, L, ...",9,...,"[4, 0, 0, 0, 2, 1]",9,7,"[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[2, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",19,16,1.203911,1.857852,3.061763
4185,00566668,4185,0.999999,0.947364,0.005535,"[4, 0, 0, 0, 0, 5]","[4, 0, 0, 0, 0, 5]","[S, L, 223, 128, L, 223, 176, L, 128, 176, L, ...","[S, L, 223, 128, L, 223, 176, L, 128, 176, L, ...",6,...,"[4, 0, 0, 0, 1, 1]",6,6,"[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",19,18,0.015733,0.982120,0.997853
6372,00852629,6372,0.999999,0.909088,0.005554,"[4, 0, 0, 0, 0, 5, 4, 2, 5]","[4, 0, 0, 0, 0, 5, 4, 2, 5, 4, 2, 5]","[S, L, 223, 128, L, 223, 151, L, 128, 151, L, ...","[S, L, 223, 128, L, 223, 152, L, 128, 152, L, ...",9,...,"[4, 0, 1, 0, 2, 2]",9,9,"[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",33,30,0.377634,1.814905,2.192538
4309,00509188,4309,0.818181,0.639998,0.005571,"[4, 0, 0, 0, 0, 4, 0, 0, 0, 0, 5]","[4, 0, 0, 0, 0, 5, 4, 0, 0, 0, 5, 5]","[S, L, 223, 128, L, 223, 203, L, 128, 203, L, ...","[S, L, 223, 128, L, 223, 207, L, 128, 207, L, ...",11,...,"[7, 0, 0, 0, 1, 1]",11,9,"[[7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[4, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",25,16,0.341757,3.679840,4.021597
5890,00294125,5890,0.500000,0.679997,0.005571,"[4, 0, 0, 0, 0, 1, 0, 0, 0, 5, 4, 0, 1, 5, 4, ...","[4, 0, 0, 0, 0, 0, 0, 0, 0, 5]","[S, L, 223, 128, L, 223, 166, L, 209, 179, L, ...","[S, L, 223, 128, L, 223, 152, L, 207, 176, L, ...",18,...,"[7, 0, 0, 0, 1, 1]",18,9,"[[7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[5, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",25,17,1.299856,3.485749,4.7856

## Segmentation: Test Metrics

In [13]:
run_name = "partseg_miou_run"

test_metrics_1 = np.load(os.path.join("..", "models", "trained_models", run_name, "test_metrics.npz"), allow_pickle=True)
test_metrics_2 = pd.read_csv(os.path.join("..", "models", "trained_models", run_name, "test_metrics.csv"))

In [14]:
test_metrics_1['class_iou'], test_metrics_1['class_acc']

(array([[0.88004574, 0.39655013, 0.23347675, 0.17249301, 0.14397624,
         0.10523883, 0.09056356, 0.0794565 , 0.11344418, 0.09527454]]),
 array([[0.95592844, 0.55201697, 0.36310666, 0.25237258, 0.22397793,
         0.15413569, 0.1289525 , 0.10086377, 0.1409144 , 0.11010391]]))

In [15]:
test_metrics_2

,mIoU,acc,mean_acc,loss
0,0.231052,0.821965,0.298237,0.696476


In [89]:
import random
random.sample([0,1,2,3,4,5,6,7,8,9],1)

[6]

In [107]:
random.randint(0, 9)

9